# Unit 13: Diagnose and improve demonstrations

Inspect applied actions and their pre-action observations. Find a timing defect before building windows. Use states 15–19 for a bounded data intervention.

**Notebook:** reference solutions. Work through the cells in order.

In [ ]:
from pathlib import Path
import os, sys
WORKSPACE = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'manifests/act_references.yaml').is_file()), None)
if WORKSPACE is None:
    raise FileNotFoundError('Run this notebook inside the VLA course workspace')
sys.path.insert(0, str(WORKSPACE / 'src'))
HF_ROOT = WORKSPACE / '.local-data' / 'hf'
os.environ.setdefault('HF_HUB_CACHE', str(HF_ROOT / 'hub'))
os.environ.setdefault('HF_DATASETS_CACHE', str(HF_ROOT / 'datasets'))
os.environ.setdefault('HF_HUB_OFFLINE', '1')
os.environ.setdefault('HF_DATASETS_OFFLINE', '1')
print('Workspace:', WORKSPACE)


In [ ]:
from __future__ import annotations
import numpy as np
from vla_lab.teaching.recording import transition_alignment, aligned_transition_indices, aligned_transition_rows, resample_aligned_transitions_10hz, dataset_tick_indices, intervention_effect
from vla_lab.teaching.data import action_window


## Exercise 1: Detect a timing defect

**Purpose:** Check that action i follows observation i on the same simulator tick.

**Contract:** `transition_alignment(observation_ticks, action_ticks) -> report`

There are N actions and N+1 observations. Observation ticks are contiguous from zero. An offset is reported in 20 Hz control ticks.

**Method:** Compare action tick i with observation tick i; do not shift the saved actions to hide an error.

**Pointers:** LIBERO controls at 20 Hz. The reference demonstration rows are 10 Hz.

<details><summary>Detailed hint</summary>

Subtract the expected pre-action tick sequence from the saved action tick sequence.

</details>

Estimated time: about 15 minutes.

In [ ]:
def transition_alignment(
    observation_ticks: Sequence[int], action_ticks: Sequence[int]
) -> dict[str, object]:
    """Check that action i was applied after observation i on the same tick."""
    obs = np.asarray(observation_ticks, dtype=np.int64)
    act = np.asarray(action_ticks, dtype=np.int64)
    if obs.ndim != 1 or act.ndim != 1 or len(obs) != len(act) + 1 or len(act) == 0:
        raise ValueError("need N action ticks and N+1 observation ticks")
    if obs[0] != 0 or np.any(np.diff(obs) != 1):
        raise ValueError("observation ticks must be contiguous and start at zero")
    expected = obs[:-1]
    offsets = act - expected
    return {
        "aligned": bool(np.array_equal(act, expected)),
        "offset_ticks": sorted(set(map(int, offsets))),
        "action_hz": 20,
        "recorded_dataset_hz": 10,
        "dataset_stride_ticks": 2,
    }

### Check

In [ ]:
report = transition_alignment([0, 1, 2, 3], [1, 2, 3])
assert report['aligned'] is False and report['offset_ticks'] == [1], f'Expected a one-tick defect, got {report}'

### Use your function

In [ ]:
print('Detected action offset (20 Hz ticks):', report['offset_ticks'])

## Exercise 2: Select aligned transitions

**Purpose:** Reject faulty row pairings before training windows are built.

**Contract:** `aligned_transition_indices(observation_ticks, action_ticks) -> integer indices`

Return indices only when all N actions align with the first N observations. Raise a clear error on a defect.

**Method:** Call the diagnostic first, then return one index for each valid action.

**Pointers:** Do not make a one-tick defect look clean by shifting every row.

<details><summary>Detailed hint</summary>

The alignment report is a guard; it does not repair the recording.

</details>

Estimated time: about 15 minutes.

In [ ]:
def aligned_transition_indices(
    observation_ticks: Sequence[int], action_ticks: Sequence[int]
) -> np.ndarray:
    """Return action row indexes paired with their pre-action observations.

    This reference path requires exact same-tick alignment. It does not shift
    rows to hide a timing defect.
    """
    report = transition_alignment(observation_ticks, action_ticks)
    if not report["aligned"]:
        raise ValueError(f"recording actions are misaligned by {report['offset_ticks']} ticks")
    return np.arange(len(action_ticks), dtype=np.int64)

### Check

In [ ]:
assert np.array_equal(aligned_transition_indices([0, 1, 2], [0, 1]), [0, 1]), 'Expected one index per aligned action'
try:
    aligned_transition_indices([0, 1, 2], [1, 2])
except ValueError:
    pass
else:
    raise AssertionError('Misaligned rows must be rejected')

### Use your function

In [ ]:
print('Aligned transitions:', aligned_transition_indices([0, 1, 2, 3], [0, 1, 2]))

## Exercise 3: Build aligned state-action rows

**Purpose:** Create the exact observations and actions used by later action windows.

**Contract:** `aligned_transition_rows(observations, actions, observation_ticks, action_ticks) -> rows`

Input has N actions, N+1 observations and matching ticks. Output row i contains observation i and action i.

**Method:** Reject a timing defect before pairing arrays.

**Pointers:** The final observation has no following action and does not become a training row.

<details><summary>Detailed hint</summary>

Reuse the checked indices from Exercise 2.

</details>

Estimated time: about 15 minutes.

In [ ]:
def aligned_transition_rows(
    observations: Sequence[object],
    actions: object,
    observation_ticks: Sequence[int],
    action_ticks: Sequence[int],
) -> list[dict[str, object]]:
    """Build state/action rows only after exact temporal alignment passes."""
    action_array = np.asarray(actions)
    indices = aligned_transition_indices(observation_ticks, action_ticks)
    if action_array.ndim != 2 or len(action_array) != len(indices):
        raise ValueError("actions must have shape (transitions, action_dim)")
    if len(observations) != len(indices) + 1:
        raise ValueError("observations must contain one final state after the last action")
    rows = []
    for index in indices:
        rows.append(
            {
                "tick": int(action_ticks[int(index)]),
                "observation": observations[int(index)],
                "action": action_array[int(index)].copy(),
            }
        )
    return rows

### Check

In [ ]:
rows = aligned_transition_rows([{'s': 0}, {'s': 1}, {'s': 2}], [[4], [5]], [0, 1, 2], [0, 1])
assert rows[1]['observation'] == {'s': 1} and np.array_equal(rows[1]['action'], [5]), 'Observation/action pairing is wrong'

### Use your function

In [ ]:
print('Second aligned transition:', rows[1])

## Exercise 4: Convert new recordings to the dataset rate

**Purpose:** Build ACT windows at the pinned 10 Hz dataset cadence from aligned 20 Hz controller transitions.

**Contract:** `resample_aligned_transitions_10hz(rows) -> selected rows`

Input rows have contiguous 20 Hz tick IDs starting at zero. Keep rows at ticks 0, 2, 4, and so on. Preserve each observation/action pair. This exercise changes only new intervention data; it does not change the historical runner.

**Method:** Select one aligned transition for each 100 ms interval. Do not average or repeat actions.

**Pointers:** Run this after alignment checks and before making ACT action windows.

<details><summary>Detailed hint</summary>

Use a stride of `control_hz // dataset_hz`, then select source indices at that stride.

</details>

Estimated time: about 15 minutes.

In [ ]:
def resample_aligned_transitions_10hz(
    rows: Sequence[dict[str, object]], *, control_hz: int = 20, dataset_hz: int = 10
) -> list[dict[str, object]]:
    """Select aligned rows at the saved 10 Hz cadence from 20 Hz control rows.

    This conversion is for newly recorded recovery data. It does not change
    the historical runner, which applies one recorded action per controller
    tick. Rows must contain contiguous control ticks beginning at zero.
    """
    if control_hz < 1 or dataset_hz < 1 or control_hz % dataset_hz:
        raise ValueError("dataset_hz must be a positive divisor of control_hz")
    if not rows:
        raise ValueError("at least one aligned transition is required")
    stride = control_hz // dataset_hz
    ticks = [int(row["tick"]) for row in rows]
    if ticks != list(range(len(rows))):
        raise ValueError("transition rows must have contiguous aligned control ticks from zero")
    return [dict(rows[index]) for index in range(0, len(rows), stride)]

### Check

In [ ]:
rows = [{'tick': i, 'observation': i, 'action': [i]} for i in range(5)]
sampled = resample_aligned_transitions_10hz(rows)
assert [row['tick'] for row in sampled] == [0, 2, 4], 'Expected aligned 10 Hz rows from the 20 Hz stream'
assert [row['action'] for row in sampled] == [[0], [2], [4]], 'Resampling must preserve applied actions'

### Use your function

In [ ]:
print('Saved data ticks:', [row['tick'] for row in sampled])

## Exercise 5: Map simulator ticks to data cadence

**Purpose:** Keep the saved 10 Hz row timing explicit.

**Contract:** `dataset_tick_indices(control_ticks) -> 10 Hz row indices`

Input is increasing even-valued 20 Hz timestamps. Output is the corresponding 10 Hz row index. It does not change how long the simulator executes each action.

**Method:** One timestamp interval in the dataset equals two simulator control ticks.

**Pointers:** Record the current mapping; do not change the pipeline timing.

<details><summary>Detailed hint</summary>

Integer-divide even control ticks by two.

</details>

Estimated time: about 15 minutes.

In [ ]:
def dataset_tick_indices(control_ticks: Sequence[int]) -> np.ndarray:
    """Map tick timestamps to 10 Hz row numbers; this does not change execution."""
    ticks = np.asarray(control_ticks, dtype=np.int64)
    if ticks.ndim != 1 or np.any(ticks < 0):
        raise ValueError("control_ticks must be a one-dimensional nonnegative sequence")
    if np.any(np.diff(ticks) <= 0):
        raise ValueError("control_ticks must increase")
    if np.any(ticks % 2):
        raise ValueError("10 Hz rows align to every second 20 Hz control tick")
    return ticks // 2

### Check

In [ ]:
assert np.array_equal(dataset_tick_indices([0, 2, 4]), [0, 1, 2]), '10 Hz mapping is wrong'

### Use your function

In [ ]:
print('dataset indices:', dataset_tick_indices([0, 2, 4, 6]))

## Exercise 6: Compare paired intervention outcomes

**Purpose:** Measure gains and losses without dropping failed attempts.

**Contract:** `intervention_effect(baseline, changed_data, *, case_ids) -> summary`

Each row has a unique `case_id` for one state and reset seed, plus `state_id`, `reset_seed`, `task_id`, and `success`. Keep the task and reset identity paired across methods. The denominator includes every attempt.

**Method:** Join by unique case ID, then check the state, seed, and task before counting gains and losses.

**Pointers:** Use intervention states 15–19. Keep baseline data, evaluation conditions, and budget fixed.

<details><summary>Detailed hint</summary>

A case key can include both state and seed, for example `state-15-seed-20260945`. Reject duplicate keys before making a lookup map.

</details>

Estimated time: about 15 minutes.

In [ ]:
def intervention_effect(
    baseline: Sequence[dict[str, object]],
    changed_data: Sequence[dict[str, object]],
    *,
    case_ids: Sequence[object],
) -> dict[str, object]:
    """Compare a data intervention on the same development cases."""
    wanted = list(case_ids)
    if not wanted or len(wanted) != len(set(wanted)):
        raise ValueError("case_ids must be nonempty and unique")
    left_ids = [row["case_id"] for row in baseline]
    right_ids = [row["case_id"] for row in changed_data]
    if len(left_ids) != len(set(left_ids)) or len(right_ids) != len(set(right_ids)):
        raise ValueError("each run must include every case exactly once")
    if any("success" not in row for row in (*baseline, *changed_data)):
        raise ValueError("each intervention result needs a success value")
    left_rows = {row["case_id"]: row for row in baseline}
    right_rows = {row["case_id"]: row for row in changed_data}
    identity_fields = ("state_id", "reset_seed", "task_id", "prompt")
    for case_id in wanted:
        a, b = left_rows[case_id], right_rows[case_id]
        present = [key for key in identity_fields if key in a or key in b]
        if any(key not in a or key not in b for key in present):
            raise ValueError("paired intervention rows need matching state, seed, task, and prompt identity")
        if any(a[key] != b[key] for key in present):
            raise ValueError("paired intervention rows differ in state, seed, task, or prompt identity")
    left = {case_id: row["success"] is True for case_id, row in left_rows.items()}
    right = {case_id: row["success"] is True for case_id, row in right_rows.items()}
    if set(left) != set(wanted) or set(right) != set(wanted):
        raise ValueError("both runs must include each selected case exactly once")
    gains = [case for case in wanted if right[case] and not left[case]]
    losses = [case for case in wanted if left[case] and not right[case]]
    return {
        "cases": len(wanted),
        "baseline_successes": sum(left.values()),
        "changed_data_successes": sum(right.values()),
        "gains": gains,
        "losses": losses,
        "failed_attempts_included": True,
    }

### Check

In [ ]:
case = 'state-15-seed-20260945'
base = {'case_id': case, 'state_id': 15, 'reset_seed': 20260945, 'task_id': 'bowl_plate', 'success': False}
changed = {**base, 'success': True}
result = intervention_effect([base], [changed], case_ids=[case])
assert result['gains'] == [case] and result['failed_attempts_included'], 'Paired gain or failure accounting is wrong'

### Use your function

In [ ]:
print('Paired intervention summary:', result)

## Larger experiment

Test whether improved demonstrations help. Record and review accepted demonstrations, then compare the frozen baseline with a bounded ACT continuation. Use held-out development states from 15–19; keep final states 20–49 closed. Record hashes, training steps, all attempts, and paired gains/losses.

Use the keyboard recorder and checked recovery pipeline. Record five to ten accepted demonstrations on development states only. This cell uses your alignment function to build state-action rows, checks a short action window, then rejects timing defects before training. The continuation uses a new optimizer and preserves the current 20 Hz recording cadence. This cell stays off by default and never fetches weights or data. See `tutorials/08-data-quality/README.md` for the keyboard recorder commands and key map.

In [ ]:
RUN_RECOVERY = False
if RUN_RECOVERY:
    from vla_lab.experiments.build_recovery_manifest import build_keyboard_manifest
    from vla_lab.experiments.recovery import continue_act_with_recovery
    from vla_lab.evaluation.states import evaluate_act_states
    from vla_lab.teaching.live import find_workspace
    from vla_lab.data.recording import RecordedEpisode
    root = find_workspace()
    recording_dir = root / '.local-data/course-results/recovery/recordings'
    recording_files = sorted(recording_dir.glob('*.npz'))
    if not 5 <= len(recording_files) <= 10:
        raise ValueError('Record and review five to ten accepted .npz episodes first')
    for path in recording_files:
        episode = RecordedEpisode.load(path)
        transitions = aligned_transition_rows(episode.observations, episode.actions,
                                               episode.observation_ticks, episode.action_ticks)
        transitions_10hz = resample_aligned_transitions_10hz(transitions)
        sampled_actions = np.stack([row['action'] for row in transitions_10hz])
        window, padding = action_window(sampled_actions, start=0, horizon=20)
        assert len(transitions) == len(episode.actions)
        assert len(transitions_10hz) == (len(episode.actions) + 1) // 2
        assert window.shape == (20, 7) and padding.shape == (20,)
    manifest_path = root / '.local-data/course-results/recovery/manifest.json'
    manifest = build_keyboard_manifest(recording_files, task_alias='bowl_plate',
                                       configs_root=root / 'configs', output=manifest_path)
    intervention_cases = list(range(15, 20))
    heldout = manifest['heldout_development_state_ids']
    if not set(intervention_cases) <= set(heldout):
        raise ValueError('Record demonstrations only on states 0–9; states 15–19 are reserved before training')
    base_checkpoint = root / '.local-data/runs/act-bowl-plate-probe-200/checkpoint-000200'
    intervention = continue_act_with_recovery(
        manifest_path=manifest_path, base_checkpoint=base_checkpoint,
        configs_root=root / 'configs', data_root=root / '.local-data',
        output_dir=root / '.local-data/course-results/recovery/continued', steps=100,
        transition_rows_fn=aligned_transition_rows,
        resample_rows_fn=resample_aligned_transitions_10hz,
        training_transition_hz=10,
        window_fn=action_window)
    heldout = intervention_cases
    baseline_dir = root / '.local-data/course-results/recovery/eval-baseline'
    changed_dir = root / '.local-data/course-results/recovery/eval-changed'
    baseline_summary = evaluate_act_states(checkpoint=base_checkpoint, configs_root=root / 'configs',
        data_root=root / '.local-data', output_dir=baseline_dir, state_ids=heldout)
    changed_summary = evaluate_act_states(checkpoint=intervention['checkpoint'], configs_root=root / 'configs',
        data_root=root / '.local-data', output_dir=changed_dir, state_ids=heldout)
    def intervention_rows(folder):
        import json
        records = [json.loads(line) for line in (folder / 'episodes.jsonl').read_text().splitlines()]
        converted = []
        for row in records:
            state_id = int(row['initial_state_id'])
            # The evaluator's frozen rule is reset_seed = 20260930 + state_id.
            # Error rows can have empty metrics; keep them with that declared seed.
            expected_seed = 20260930 + state_id
            saved_seed = row.get('metrics', {}).get('reset_seed')
            if saved_seed is not None and int(saved_seed) != expected_seed:
                raise ValueError(f"Saved reset seed differs for state {state_id}")
            converted.append({'case_id': f"state-{state_id}-seed-{expected_seed}",
                 'state_id': state_id, 'reset_seed': expected_seed,
                 'task_id': row['task_id'], 'success': row['success'] is True})
        return converted
    expected_cases = [f'state-{state}-seed-{20260930 + state}' for state in heldout]
    result = intervention_effect(intervention_rows(baseline_dir), intervention_rows(changed_dir),
                                 case_ids=expected_cases)
    (root / '.local-data/course-results/recovery/paired-summary.json').write_text(json.dumps(result, indent=2) + '\n')
    print('New checkpoint:', intervention['checkpoint'], 'paired result:', result)
    print('Baseline attempts/successes:', baseline_summary['attempts'], baseline_summary['successes'])
    print('Changed attempts/successes:', changed_summary['attempts'], changed_summary['successes'])

## Record your result

Keep the inputs, every attempt, and the result table with the notebook. A failed run is a result and stays in the denominator.